# T04 — Resources로 재료 물성치 노출

**학습 목표**
- `@mcp.resource()` 데코레이터로 정적/동적 리소스 정의
- URI 스킴 설계 (`config://`, `data://`)
- 한국 콘크리트·철강·철근·목재 물성치를 LLM에 제공
- T03의 `SimpleMCPClient`로 리소스 읽기

**Prerequisites**
- T01 완료 (`tutorial_server.py` 도구 등록)
- T02 완료 (Inspector 검증 경험)
- T03 완료 (`SimpleMCPClient` 구현)

> [!ref] 강의 노트: `Week_07.md` §2.2-§2.3 (Resources 정의/접근)

## §0. 강의노트 매핑

> 📖 **Week_07.md §2.2-§2.3** (Resources 정의/접근)

| 측면 | Skilljar 원본 (`skilljar/S6_04`) | 본 튜토리얼 (T04) |
|---|---|---|
| 도메인 | 문서 컬렉션 (`docs://documents`) | 한국 건축공학 재료 (`data://materials/...`) |
| 패턴 | direct + templated | direct + templated (동일 패턴) |

URI 스킴은 다르지만 **`@mcp.resource()` 패턴은 동일**. 한 번 익히면 어떤 도메인이든 적용 가능.

> [!tip] 본 튜토리얼은 학생 친화 버전입니다. 실제 강의 제출은 `skilljar/S6_04` 또는 `structural/S6_04` 권장.

## §1. Setup

In [ ]:
# ── Setup ──────────────────────────────────────────────
import json
import asyncio
from mcp.server.fastmcp import FastMCP

# T01에서 생성한 서버 인스턴스를 확장한다고 가정
# (실제 코드에서는 tutorial_server.py를 import해도 됨)
mcp = FastMCP("TutorialMCP", log_level="ERROR")

print("FastMCP 인스턴스 준비 완료:", mcp.name)

## §2. Resources vs Tools — 어떻게 구분하는가?

| 구분 | Tools | Resources |
|------|-------|----------|
| 목적 | **함수 실행** (계산, API 호출) | **데이터 제공** (파일, DB, 기준 문서) |
| 접근 | 함수 이름으로 호출 | **URI로 접근** |
| 제어 | 모델(LLM)이 호출 결정 | 애플리케이션이 접근 결정 |
| 변경 | 상태 변경 가능 | **읽기 전용** |

> [!finding] 한국 건축공학 예시
> - **Resource**: KDS 41 17 00 §4.3 조문 (변하지 않는 기준)
> - **Tool**: 휨 강도 검토 계산 함수 (입력 받아 결과 반환)

## §3. 정적 리소스 — 앱 설정

In [ ]:
# 정적 리소스 1: 앱 설정
@mcp.resource("config://app/settings")
def get_app_settings() -> str:
    """애플리케이션 설정을 반환합니다."""
    return json.dumps({
        "version": "1.0",
        "debug": False,
        "max_connections": 100,
        "locale": "ko_KR"
    }, indent=2, ensure_ascii=False)

print("config://app/settings 등록 완료")

## §4. 정적 리소스 — 콘크리트 물성치

한국 KS F 2403 / KDS 14 20 10 기준 콘크리트 강도 등급별 물성치.

탄성계수 공식: $E_c = 8500 \cdot (f_{ck})^{1/3}$ MPa (KCI 2017)

In [ ]:
# 정적 리소스 2: 콘크리트 물성치
@mcp.resource("data://materials/concrete")
def get_concrete_properties() -> str:
    """한국 KCI 2017 기준 콘크리트 강도 등급별 물성치."""
    return json.dumps({
        "standard": "KDS 14 20 10",
        "Ec_formula": "Ec = 8500 * (fck)^(1/3) MPa",
        "unit_weight": "24 kN/m3",
        "grades": {
            "C24": {"fck": 24, "Ec": 25742},
            "C27": {"fck": 27, "Ec": 26871},
            "C30": {"fck": 30, "Ec": 27924},
            "C35": {"fck": 35, "Ec": 29388},
            "C40": {"fck": 40, "Ec": 30722}
        }
    }, indent=2, ensure_ascii=False)

print("data://materials/concrete 등록 완료")

> ☑ **체크포인트 1**: 정적 리소스 2개 등록
>
> 다음 셀에서 `await mcp.list_resources()`를 호출하면 위 두 개의 URI가 보여야 합니다.

## §5. Templated 리소스 — 동적 재료 조회

URI에 `{material_type}` 파라미터를 포함하면 재료 종류별로 동적 조회 가능.

In [ ]:
# Templated 리소스: 재료별 물성치
@mcp.resource("data://materials/{material_type}")
def get_material_properties(material_type: str) -> str:
    """지정된 재료의 물성치를 반환합니다.

    Args:
        material_type: 재료 종류 (steel, rebar, wood)
    """
    materials = {
        "steel": {
            "standard": "KS D 3503 / KDS 14 31 05",
            "type": "structural_steel",
            "Es": 200000,
            "unit": "MPa",
            "grades": {
                "SS275": {"Fy": 275, "Fu": 410},
                "SS355": {"Fy": 355, "Fu": 490},
                "SM490": {"Fy": 315, "Fu": 490}
            }
        },
        "rebar": {
            "standard": "KS D 3504",
            "type": "reinforcing_bar",
            "Es": 200000,
            "unit": "MPa",
            "grades": {
                "SD400": {"fy": 400, "fu": 560},
                "SD500": {"fy": 500, "fu": 620}
            }
        }
    }
    if material_type not in materials:
        return json.dumps({"error": f"'{material_type}' 재료를 찾을 수 없음"}, ensure_ascii=False)
    return json.dumps(materials[material_type], indent=2, ensure_ascii=False)

print("data://materials/{material_type} 등록 완료 (steel, rebar)")

## §6. 확장 — 목재 추가

한국 라디아타 송 (Pinus radiata) 구조용 목재 등급 추가.

> [!action] 학생 실습: 위 함수에 `wood` 케이스를 직접 추가해보세요.

In [ ]:
# 위 함수를 wood 지원 버전으로 재정의
@mcp.resource("data://materials/{material_type}")
def get_material_properties_v2(material_type: str) -> str:
    """한국 구조용 재료 물성치 (wood 추가)."""
    materials = {
        "steel": {
            "grades": {"SS275": {"Fy": 275}, "SS355": {"Fy": 355}, "SM490": {"Fy": 315}},
            "Es": 200000, "unit": "MPa"
        },
        "rebar": {
            "grades": {"SD400": {"fy": 400}, "SD500": {"fy": 500}},
            "Es": 200000, "unit": "MPa"
        },
        "wood": {
            "standard": "KDS 41 50 00 (구조용 목재)",
            "species": "Pinus radiata (라디아타 송)",
            "grades": {
                "E10": {"E": 10000, "fb": 14, "fc": 8},
                "E12": {"E": 12000, "fb": 18, "fc": 10}
            },
            "unit": "MPa"
        }
    }
    return json.dumps(materials.get(material_type, {"error": f"'{material_type}' not found"}),
                      indent=2, ensure_ascii=False)

print("wood 지원 버전 등록 완료")

> ☑ **체크포인트 2**: Templated 리소스 + wood 확장
>
> 이제 4가지 재료 (concrete=정적, steel/rebar/wood=동적) 모두 조회 가능.

## §7. 리소스 목록·읽기 검증

In [ ]:
async def demo_resources():
    # 1. 정적 리소스 목록
    resources = await mcp.list_resources()
    print("=== 정적 리소스 ===")
    for r in resources:
        print(f"  URI: {r.uri}")
        print(f"  이름: {r.name}")
    print()

    # 2. 리소스 템플릿 목록
    templates = await mcp.list_resource_templates()
    print("=== Templated 리소스 ===")
    for t in templates:
        print(f"  URI 템플릿: {t.uriTemplate}")
        print(f"  이름: {t.name}")
    print()

    # 3. 콘크리트 물성치 읽기
    print("=== 콘크리트 (정적) ===")
    content = await mcp.read_resource("data://materials/concrete")
    print(content[0].content[:300] + "..." if hasattr(content[0], 'content') else str(content)[:300])
    print()

    # 4. 철강 물성치 읽기 (templated)
    print("=== 철강 (templated) ===")
    steel = await mcp.read_resource("data://materials/steel")
    print(steel[0].content[:300] + "..." if hasattr(steel[0], 'content') else str(steel)[:300])

await demo_resources()

## §8. MCPClient로 접근 (T03의 `SimpleMCPClient` 활용)

T03에서 만든 클라이언트에 `read_resource()` 메서드를 추가하면 외부 프로세스에서도 리소스 접근 가능.

In [ ]:
# T03의 SimpleMCPClient에 read_resource 메서드 추가 예시
from contextlib import AsyncExitStack
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

class SimpleMCPClient:
    """T03 클라이언트 + read_resource 추가."""
    def __init__(self, command: str, args: list):
        self.params = StdioServerParameters(command=command, args=args)
        self.exit_stack = AsyncExitStack()
        self.session: ClientSession | None = None

    async def __aenter__(self):
        transport = await self.exit_stack.enter_async_context(stdio_client(self.params))
        read, write = transport
        self.session = await self.exit_stack.enter_async_context(ClientSession(read, write))
        await self.session.initialize()
        return self

    async def __aexit__(self, *args):
        await self.exit_stack.aclose()

    async def list_resources(self):
        return (await self.session.list_resources()).resources

    async def read_resource(self, uri: str):
        """NEW in T04: URI로 리소스 콘텐츠 읽기."""
        result = await self.session.read_resource(uri)
        return result.contents[0].text if result.contents else None

# 사용 예시 (서버 파일이 있다고 가정):
# async with SimpleMCPClient("python", ["tutorial_server.py"]) as client:
#     concrete = await client.read_resource("data://materials/concrete")
#     print(concrete)
print("SimpleMCPClient 클래스 정의 완료 (read_resource 메서드 추가)")

## §9. Skilljar `docs://` 와의 비교

같은 패턴, 다른 도메인. 두 도메인을 비교 학습하면 어떤 도메인에서도 응용 가능.

| 측면 | Skilljar (`skilljar/S6_04`) | Tutorial (T04, 본 노트북) |
|---|---|---|
| 도메인 | 문서 컬렉션 | 건축 재료 물성치 |
| Direct URI | `docs://documents` (모든 문서 ID) | `data://materials/concrete` (콘크리트 단일) |
| Templated URI | `docs://documents/{doc_id}` | `data://materials/{material_type}` |
| MIME types | `application/json`, `text/plain` | `application/json` (단일) |
| 데이터 출처 | 메모리 dict | KS/KDS 표준 (정적 데이터) |

> [!tip] **공통 패턴**
> 1. URI 스킴 설계 (`scheme://path/{param}`)
> 2. `@mcp.resource(uri)` 데코레이터로 함수 등록
> 3. 정적은 `list_resources()`, 동적은 `list_resource_templates()`로 노출
> 4. 클라이언트는 `read_resource(uri)`로 호출

## §10. `tutorial_server.py` 갱신 저장

T01의 도구 + T04의 리소스를 모두 포함한 서버 파일 생성.

In [ ]:
server_code = '''# tutorial_server.py — T01 도구 + T04 리소스 통합
import json
from datetime import datetime
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("TutorialMCP")

# ── Tools (T01) ──────────────────────────────────────────
@mcp.tool()
def get_current_time(format: str = "%Y-%m-%d %H:%M:%S") -> str:
    """현재 날짜와 시간을 반환합니다."""
    return datetime.now().strftime(format)

@mcp.tool()
def add_numbers(a: float, b: float) -> float:
    """두 숫자를 더합니다."""
    return a + b

# ── Resources (T04) ──────────────────────────────────────
@mcp.resource("config://app/settings")
def get_app_settings() -> str:
    return json.dumps({"version": "1.0", "locale": "ko_KR"}, indent=2)

@mcp.resource("data://materials/concrete")
def get_concrete() -> str:
    return json.dumps({
        "C24": {"fck": 24, "Ec": 25742},
        "C27": {"fck": 27, "Ec": 26871},
        "C30": {"fck": 30, "Ec": 27924}
    }, indent=2)

@mcp.resource("data://materials/{material_type}")
def get_material(material_type: str) -> str:
    db = {
        "steel": {"SS275": {"Fy": 275}, "SS355": {"Fy": 355}},
        "rebar": {"SD400": {"fy": 400}, "SD500": {"fy": 500}},
        "wood":  {"E10": {"E": 10000, "fb": 14}}
    }
    return json.dumps(db.get(material_type, {}), indent=2)

if __name__ == "__main__":
    mcp.run()
'''

with open("tutorial_server.py", "w", encoding="utf-8") as f:
    f.write(server_code)

print("tutorial_server.py 저장 완료 (도구 2 + 리소스 2 + 템플릿 1)")

> ☑ **체크포인트 3**: `tutorial_server.py` 갱신 완료
>
> 다음과 같이 검증 가능:
> ```bash
> mcp dev tutorial_server.py   # Inspector로 시각 확인
> ```

## §11. 트러블슈팅

| 증상 | 원인 | 해결 |
|---|---|---|
| `ResourceNotFound` | URI 오타 또는 미등록 | `await mcp.list_resources()`로 등록 확인 |
| MIME type 누락 경고 | `mime_type` 파라미터 미지정 | `@mcp.resource(uri, mime_type="application/json")` 추가 |
| Templated URI가 list에 안 보임 | `list_resources()`가 아닌 `list_resource_templates()` 사용 필요 | 템플릿 전용 API 호출 |
| `ValueError: Path conflict` | 동일 URI를 중복 등록 | URI 스킴 재설계 (e.g. `data://materials/v2/...`) |

## §12. 다음 단계

- **T05**: Prompts로 KDS 검토 시나리오 외부화
- **T06**: Capstone — 자율 도메인 (개인 KB, 할 일, 레시피 등)

> [!ref] **현재 진행 상황**
> - ✅ T01: Tools 정의
> - ✅ T02: Inspector 검증
> - ✅ T03: SimpleMCPClient
> - ✅ T04: Resources (현재 노트북)
> - ⏳ T05: Prompts
> - ⏳ T06: Capstone